## ***Reseau de neuronnes***

## 1. Structure des données (panel temporel)

On considère un panel indexé par :
- $i \in \{1,\dots,N\}$ : unité spatiale (`NUMERO`)
- $t \in \mathbb{Z}$ : temps (`ANNEE`)
- $m \in \{1,\dots,12\}$ : mois

La variable cible est le **SWI uniforme mensuel** :

$$
\mathbf{Y}_{i,t}
=
\begin{pmatrix}
\text{SWI}^{\text{unif}}_{i,t,1} \\
\vdots \\
\text{SWI}^{\text{unif}}_{i,t,12}
\end{pmatrix}
\in \mathbb{R}^{12}
$$



## 2. Variables explicatives

### 2.1 Variables auto-régressives (lags)

Pour chaque mois $m$ :

- Lags de SWI uniforme ($L_{\text{unif}}=2$) :
$$
\left\{
\text{SWI}^{\text{unif}}_{i,t-\ell,m}
\right\}_{\ell=1}^{2}
$$

- Lags de SWI standard ($L_{\text{SWI}}=4$) :
$$
\left\{
\text{SWI}_{i,t-\ell,m}
\right\}_{\ell=1}^{4}
$$



### 2.2 Variables climatiques retardées

Pour chaque variable climatique $Z \in \mathcal{Z}$ et chaque mois $m$ :

$$
Z_{i,t-1,m}
$$

où $\mathcal{Z}$ regroupe température, humidité, précipitations, neige et écoulement.



### 2.3 Variables statiques

Une variable constante dans le temps :

$$
\text{argile\_niveau}_i
$$



## 3. Vecteur des features

Le vecteur explicatif complet est :

$$
\mathbf{X}_{i,t}
=
\begin{pmatrix}
\text{SWI}^{\text{unif}}_{i,t-1,\cdot} \\
\text{SWI}^{\text{unif}}_{i,t-2,\cdot} \\
\text{SWI}_{i,t-1,\cdot} \\
\vdots \\
\text{SWI}_{i,t-4,\cdot} \\
\mathcal{Z}_{i,t-1,\cdot} \\
\text{argile\_niveau}_i
\end{pmatrix}
\in \mathbb{R}^{p}
$$



## 4. Standardisation des variables

Les variables explicatives sont standardisées sur l’échantillon d’apprentissage :

$$
\tilde{X}_{i,t,j}
=
\frac{X_{i,t,j} - \mu_j}{\sigma_j}
$$

où $(\mu_j,\sigma_j)$ sont estimés sur le train uniquement.



## 5. Modèle de réseau de neurones multi-sortie

Le modèle est une fonction non linéaire :

$$
\hat{\mathbf{Y}}_{i,t}
=
f_\theta(\tilde{\mathbf{X}}_{i,t})
$$



### 5.1 Architecture du réseau

Le réseau est défini par :

$$
f_\theta
=
W_3
\circ
\text{ReLU}
\circ
W_2
\circ
\text{ReLU}
\circ
W_1
$$

avec :

$$
\begin{aligned}
\mathbf{h}_1 &= \text{ReLU}(W_1 \tilde{\mathbf{X}} + b_1) \\
\mathbf{h}_2 &= \text{ReLU}(W_2 \mathbf{h}_1 + b_2) \\
\hat{\mathbf{Y}} &= W_3 \mathbf{h}_2 + b_3
\end{aligned}
$$



## 6. Fonction de perte

Le modèle est entraîné par minimisation de la perte quadratique moyenne multi-sortie :

$$
\mathcal{L}(\theta)
=
\frac{1}{N}
\sum_{i,t}
\left\|
\mathbf{Y}_{i,t}
-
\hat{\mathbf{Y}}_{i,t}
\right\|_2^2
$$



## 7. Optimisation

L’optimisation est réalisée par Adam avec pénalisation $L_2$ :

$$
\theta^{(k+1)}
=
\theta^{(k)}
-
\eta
\left(
\nabla_\theta \mathcal{L}
+
\lambda \theta
\right)
$$



## 8. Métriques d’évaluation

### 8.1 Métriques globales

$$
\text{RMSE}_{\text{global}}
=
\sqrt{
\frac{1}{12N}
\sum_{i,t,m}
\left(
Y_{i,t,m}
-
\hat{Y}_{i,t,m}
\right)^2
}
$$

$$
\text{MAE}_{\text{global}}
=
\frac{1}{12N}
\sum_{i,t,m}
\left|
Y_{i,t,m}
-
\hat{Y}_{i,t,m}
\right|
$$



### 8.2 Métriques par mois

Pour chaque mois $m$ :

$$
\text{RMSE}_m
=
\sqrt{
\frac{1}{N}
\sum_{i,t}
\left(
Y_{i,t,m}
-
\hat{Y}_{i,t,m}
\right)^2
}
$$

$$
\text{MAE}_m
=
\frac{1}{N}
\sum_{i,t}
\left|
Y_{i,t,m}
-
\hat{Y}_{i,t,m}
\right|
$$



## 9. Résumé du modèle

Le modèle est un **réseau de neurones feedforward multi-sortie** appliqué à des données de panel temporel, combinant :
- des composantes auto-régressives multivariées,
- des variables climatiques retardées,
- une non-linéarité capturée par des couches ReLU.


In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np


In [2]:
from pathlib import Path
import pandas as pd

# Dossier où se trouve le notebook
current_dir = Path.cwd()

# Parent du parent
root_dir = current_dir.parent.parent

# Fichier à importer
file_path = root_dir / "data" / "data_cleaned.csv"

# Import
df = pd.read_csv(file_path)


In [38]:
dup_par_annee = (
    df.groupby(["ANNEE", "NUMERO"])
      .size()
      .reset_index(name="n")
      .query("n > 1")
      .sort_values(["ANNEE", "NUMERO"])
)

dup_par_annee


,ANNEE,NUMERO,n
4,2000,6,5
12,2000,19,3
21,2000,33,2
32,2000,47,4
45,2000,62,3
...,...,...,...
224253,2024,9611,5
224254,2024,9612,2
224292,2024,9650,3
224293,2024,9655,3


In [40]:
import pandas as pd

# ================================
# 1) Identifier les NUMERO dupliqués par année
# ================================
dup_counts = (
    df.groupby(["ANNEE", "NUMERO"])
      .size()
      .reset_index(name="n")
)

# On garde uniquement ceux qui se répètent (n > 1)
dup_only = dup_counts[dup_counts["n"] > 1]

# ================================
# 2) Est-ce les mêmes NUMERO dupliqués chaque année ?
# ================================
numeros_dupliques_par_annee = (
    dup_only.groupby("ANNEE")["NUMERO"]
            .apply(set)
)

annee_ref = numeros_dupliques_par_annee.index.min()
ref_set = numeros_dupliques_par_annee.loc[annee_ref]

comparaison_points = pd.DataFrame({
    "ANNEE": numeros_dupliques_par_annee.index,
    "meme_NUMERO_dupliques_que_ref": numeros_dupliques_par_annee.apply(lambda x: x == ref_set),
    "nb_NUMERO_dupliques": numeros_dupliques_par_annee.apply(len),
    "nb_manquants_vs_ref": numeros_dupliques_par_annee.apply(lambda x: len(ref_set - x)),
    "nb_en_plus_vs_ref": numeros_dupliques_par_annee.apply(lambda x: len(x - ref_set))
}).reset_index(drop=True)

# ================================
# 3) Est-ce le même nombre de répétitions chaque année ?
# ================================
counts_wide = (
    dup_only.pivot(index="NUMERO", columns="ANNEE", values="n")
)

stabilite_repetitions = pd.DataFrame({
    "min_repetitions": counts_wide.min(axis=1),
    "max_repetitions": counts_wide.max(axis=1),
    "repetitions_stables": counts_wide.min(axis=1) == counts_wide.max(axis=1)
})

# ================================
# 4) Diagnostic final
# ================================
diagnostic_final = {
    "meme_points_dupliques_chaque_annee": comparaison_points["meme_NUMERO_dupliques_que_ref"].all(),
    "meme_nombre_de_repetitions_chaque_annee": stabilite_repetitions["repetitions_stables"].all(),
    "nb_NUMERO_dupliques_instables": (~stabilite_repetitions["repetitions_stables"]).sum()
}

# ================================
# OUTPUT
# ================================
print("=== NUMERO dupliqués : comparaison par année ===")
print(comparaison_points)

print("\n=== Stabilité du nombre de répétitions par NUMERO ===")
print(stabilite_repetitions)

print("\n=== Diagnostic final ===")
print(diagnostic_final)


=== NUMERO dupliqués : comparaison par année ===
    ANNEE  meme_NUMERO_dupliques_que_ref  nb_NUMERO_dupliques  \
0    2000                           True                   97   
1    2001                           True                   97   
2    2002                           True                   97   
3    2003                           True                   97   
4    2004                           True                   97   
5    2005                           True                   97   
6    2006                           True                   97   
7    2007                           True                   97   
8    2008                           True                   97   
9    2009                           True                   97   
10   2010                           True                   97   
11   2011                           True                   97   
12   2012                           True                   97   
13   2013                           True 

In [4]:
df.head()


,NUMERO,LAMBX_93,LAMBY_93,LAMBX_2,LAMBY_2,ANNEE,code_insee_commune,commune,code_insee_de_la_region,region,...,WG_RACINE_03,WG_RACINE_04,WG_RACINE_05,WG_RACINE_06,WG_RACINE_07,WG_RACINE_08,WG_RACINE_09,WG_RACINE_10,WG_RACINE_11,WG_RACINE_12
0,1860,111467,6838785,60000,2401000,2000,29084,Île-Molène,53,Bretagne,...,0.30,0.30,0.30,0.29,0.28,0.26,0.25,0.28,0.31,0.32
1,2477,127137,6798687,76000,2361000,2000,29168,Plogoff,53,Bretagne,...,0.23,0.24,0.23,0.22,0.19,0.18,0.18,0.21,0.25,0.27
2,1984,127392,6830663,76000,2393000,2000,29190,Plougonvelin,53,Bretagne,...,0.29,0.29,0.29,0.27,0.25,0.23,0.23,0.26,0.30,0.32
3,1861,127455,6838657,76000,2401000,2000,29201,Ploumoguer,53,Bretagne,...,0.30,0.31,0.30,0.29,0.27,0.25,0.24,0.28,0.32,0.33
4,1737,127519,6846652,76000,2409000,2000,29098,Lampaul-Plouarzel,53,Bretagne,...,0.30,0.31,0.30,0.29,0.27,0.24,0.24,0.28,0.32,0.33


## Implémentation des Reseau de neuronnes

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# =========================================================
# 1) Colonnes
# =========================================================
SWI_UNIF_COLS = [f"swi_unif_{m:02d}" for m in range(1, 13)]
SWI_COLS      = [f"SWI_{m:02d}" for m in range(1, 13)]

TARGET_COLS = SWI_UNIF_COLS
X_BASE_COLS = SWI_UNIF_COLS + SWI_COLS   # 24 variables

# =========================================================
# 2) Création des lags (panel)
# =========================================================
def make_lags_panel(df, cols, lags, group_col="NUMERO", time_col="ANNEE"):
    df = df.sort_values([group_col, time_col]).copy()

    for col in cols:
        for l in range(1, lags + 1):
            df[f"{col}_lag{l}"] = (
                df.groupby(group_col)[col].shift(l)
            )

    return df.dropna().reset_index(drop=True)

# =========================================================
# 3) Sélection des colonnes X pour une paire (L_unif, L_swi)
# =========================================================
def build_X_cols_pair(L_unif, L_swi):
    cols = []

    # swi_unif lags
    for l in range(1, L_unif + 1):
        cols += [f"swi_unif_{m:02d}_lag{l}" for m in range(1, 13)]

    # SWI lags
    for l in range(1, L_swi + 1):
        cols += [f"SWI_{m:02d}_lag{l}" for m in range(1, 13)]

    return cols

# =========================================================
# 4) Réseau de neurones multi-output
# =========================================================
class MultiOutputNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# =========================================================
# 5) Entraînement + RMSE
# =========================================================
def train_and_eval_rmse(
    X_train, Y_train,
    X_val, Y_val,
    epochs=500,
    lr=1e-3,
    weight_decay=1e-4,
    verbose=False
):
    Xtr = torch.tensor(X_train, dtype=torch.float32)
    Ytr = torch.tensor(Y_train, dtype=torch.float32)
    Xva = torch.tensor(X_val,   dtype=torch.float32)
    Yva = torch.tensor(Y_val,   dtype=torch.float32)

    model = MultiOutputNN(Xtr.shape[1], Ytr.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(Xtr), Ytr)
        loss.backward()
        optimizer.step()

        if verbose and epoch % 100 == 0:
            with torch.no_grad():
                val_rmse = torch.sqrt(loss_fn(model(Xva), Yva))
            print(
                f"Epoch {epoch:4d} | "
                f"Train MSE = {loss.item():.5f} | "
                f"Val RMSE = {val_rmse.item():.5f}"
            )

    with torch.no_grad():
        pred = model(Xva)
        rmse = torch.sqrt(((pred - Yva) ** 2).mean()).item()

    return rmse

# =========================================================
# 6) Sélection conjointe (L_unif, L_swi)
# =========================================================
def select_best_L_pair(
    df,
    L_MAX=10,
    train_end=2015,
    val_end=2019,
    group_col="NUMERO",
    time_col="ANNEE",
    epochs=500
):
    # Création des lags une seule fois
    df_lag = make_lags_panel(
        df,
        cols=X_BASE_COLS,
        lags=L_MAX,
        group_col=group_col,
        time_col=time_col
    )

    # Targets
    Y = df_lag[TARGET_COLS].values.astype(np.float32)

    # Split temporel
    years = df_lag[time_col].values
    train_mask = years <= train_end
    val_mask   = (years > train_end) & (years <= val_end)

    rmse_grid = {}

    for L_unif in range(1, L_MAX + 1):
        for L_swi in range(1, L_MAX + 1):

            X_cols = build_X_cols_pair(L_unif, L_swi)
            X = df_lag[X_cols].values.astype(np.float32)

            X_train, Y_train = X[train_mask], Y[train_mask]
            X_val,   Y_val   = X[val_mask],   Y[val_mask]

            print(
                f"\n===== L_unif={L_unif}, L_swi={L_swi} | "
                f"features={X_train.shape[1]} ====="
            )

            rmse = train_and_eval_rmse(
                X_train, Y_train,
                X_val, Y_val,
                epochs=epochs,
                verbose=True
            )

            rmse_grid[(L_unif, L_swi)] = rmse

            print(
                f"L_unif={L_unif}, L_swi={L_swi} | RMSE={rmse:.5f}"
            )

    best_pair = min(rmse_grid, key=rmse_grid.get)

    return rmse_grid, best_pair

# =========================================================
# 7) LANCEMENT
# =========================================================
rmse_grid, (best_L_unif, best_L_swi) = select_best_L_pair(
    df,
    L_MAX=10,
    train_end=2022,
    val_end=2024,
    epochs=1000
)

print(
    f"\n BEST PAIR : "
    f"L_unif={best_L_unif}, "
    f"L_swi={best_L_swi} | "
    f"RMSE={rmse_grid[(best_L_unif, best_L_swi)]:.5f}"
)



===== L_unif=1, L_swi=1 | features=24 =====
Epoch    0 | Train MSE = 0.47747 | Val RMSE = 0.71476
Epoch  100 | Train MSE = 0.03090 | Val RMSE = 0.19763
Epoch  200 | Train MSE = 0.02431 | Val RMSE = 0.18376
Epoch  300 | Train MSE = 0.02113 | Val RMSE = 0.19182
Epoch  400 | Train MSE = 0.01952 | Val RMSE = 0.20113
Epoch  500 | Train MSE = 0.01841 | Val RMSE = 0.19968
Epoch  600 | Train MSE = 0.01766 | Val RMSE = 0.20101
Epoch  700 | Train MSE = 0.01702 | Val RMSE = 0.20357
Epoch  800 | Train MSE = 0.01647 | Val RMSE = 0.20250
Epoch  900 | Train MSE = 0.01612 | Val RMSE = 0.20180
L_unif=1, L_swi=1 | RMSE=0.20723

===== L_unif=1, L_swi=2 | features=36 =====
Epoch    0 | Train MSE = 0.40235 | Val RMSE = 0.66164
Epoch  100 | Train MSE = 0.02889 | Val RMSE = 0.20769
Epoch  200 | Train MSE = 0.02152 | Val RMSE = 0.19909
Epoch  300 | Train MSE = 0.01777 | Val RMSE = 0.20343
Epoch  400 | Train MSE = 0.01600 | Val RMSE = 0.21602
Epoch  500 | Train MSE = 0.01487 | Val RMSE = 0.22248
Epoch  600 | 

- La meilleur paire est $(L_{unif} = 2$ et $L_{swi} = 4)$

In [4]:
# =========================================================
# Réseau de neurones multi-output (12 mois) + évaluation
# - Split temporel strict
# - Standardisation des features (X)
# - Entraînement PyTorch
# - Prédictions + RMSE/MAE globales + par mois
# =========================================================

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error


# =========================================================
# 0) PARAMÈTRES
# =========================================================
GROUP_COL = "NUMERO"
TIME_COL  = "ANNEE"

# Lags choisis
L_UNIF = 2
L_SWI  = 4
L_OTHER = 1

# Split temporel
TRAIN_END = 2019      # train: <= TRAIN_END
TEST_START = 2020     # test : >= TEST_START

# Training NN
BATCH_SIZE = 256
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-4
PRINT_EVERY = 20
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)


# =========================================================
# 1) LISTES DE VARIABLES 
# =========================================================
SWI_UNIF_COLS = [f"swi_unif_{str(m).zfill(2)}" for m in range(1, 13)]
SWI_COLS      = [f"SWI_{str(m).zfill(2)}" for m in range(1, 13)]

THERMAL_COLS  = [f"T_{str(m).zfill(2)}" for m in range(1, 13)] #+ \
                #[f"TINF_H_{str(m).zfill(2)}" for m in range(1, 13)] + \
                #[f"TSUP_H_{str(m).zfill(2)}" for m in range(1, 13)]

HUMIDITY_COLS = [f"HU_{str(m).zfill(2)}" for m in range(1, 13)]

PRECIP_COLS   = [f"PRELIQ_{str(m).zfill(2)}" for m in range(1, 13)] + \
                [f"PRENEI_{str(m).zfill(2)}" for m in range(1, 13)]

EVAP_COLS     = [f"EVAP_{str(m).zfill(2)}" for m in range(1, 13)] #+ \
                #[f"ETP_{str(m).zfill(2)}" for m in range(1, 13)]

SNOW_COLS     = [f"HTEURNEIGE_{str(m).zfill(2)}" for m in range(1, 13)] + \
                [f"SNOW_FRAC_{str(m).zfill(2)}" for m in range(1, 13)]

FLOW_COLS     = [f"ECOULEMENT_{str(m).zfill(2)}" for m in range(1, 13)] + \
                [f"Q_{str(m).zfill(2)}" for m in range(1, 13)]

STATIC_COLS   = ["argile_niveau"]

TARGET_COLS = SWI_UNIF_COLS


# =========================================================
# 2) FONCTION: AJOUT DES LAGS (panel)
# =========================================================
def add_lags_panel(df, cols, lags, group_col=GROUP_COL, time_col=TIME_COL):
    out = df.sort_values([group_col, time_col]).copy()
    for col in cols:
        for l in range(1, lags + 1):
            out[f"{col}_lag{l}"] = out.groupby(group_col)[col].shift(l)
    return out


# =========================================================
# 3) BUILD FEATURES COLS X
# =========================================================
def build_feature_columns(L_unif, L_swi, other_cols, static_cols):
    X_cols = []

    # swi_unif lags
    for l in range(1, L_unif + 1):
        X_cols += [f"{c}_lag{l}" for c in SWI_UNIF_COLS]

    # SWI lags
    for l in range(1, L_swi + 1):
        X_cols += [f"{c}_lag{l}" for c in SWI_COLS]

    # autres variables lag1
    X_cols += [f"{c}_lag1" for c in other_cols]

    # statiques en niveau
    X_cols += static_cols

    return X_cols


# =========================================================
# 4) RÉSEAU DE NEURONES
# =========================================================
class MultiOutputNN(nn.Module):
    def __init__(self, input_dim, output_dim=12):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)


# =========================================================
# 5) ENTRAÎNEMENT + EVAL
# =========================================================
def evaluate_metrics(y_true, y_pred):
    # global (flatten)
    rmse_global = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
    mae_global  = mean_absolute_error(y_true.flatten(), y_pred.flatten())

    # par mois
    rmse_by_month = {m+1: np.sqrt(mean_squared_error(y_true[:, m], y_pred[:, m])) for m in range(12)}
    mae_by_month  = {m+1: mean_absolute_error(y_true[:, m], y_pred[:, m]) for m in range(12)}

    return rmse_global, mae_global, rmse_by_month, mae_by_month


def train_nn_and_report(X_train, Y_train, X_test, Y_test):
    # Standardisation X
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # DataLoaders
    train_ds = TensorDataset(
        torch.tensor(X_train_sc, dtype=torch.float32),
        torch.tensor(Y_train, dtype=torch.float32)
    )
    test_ds = TensorDataset(
        torch.tensor(X_test_sc, dtype=torch.float32),
        torch.tensor(Y_test, dtype=torch.float32)
    )
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    # Model
    model = MultiOutputNN(input_dim=X_train_sc.shape[1], output_dim=Y_train.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()

    # Train loop
    for epoch in range(EPOCHS):
        model.train()
        losses = []

        for xb, yb in train_loader:
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        # periodic eval
        if (epoch % PRINT_EVERY == 0) or (epoch == EPOCHS - 1):
            model.eval()
            with torch.no_grad():
                preds = []
                ys = []
                for xb, yb in test_loader:
                    preds.append(model(xb))
                    ys.append(yb)
                preds = torch.cat(preds).cpu().numpy()
                ys = torch.cat(ys).cpu().numpy()

            rmse_g, mae_g, _, _ = evaluate_metrics(ys, preds)
            print(f"Epoch {epoch:03d} | Train MSE={np.mean(losses):.5f} | Test RMSE={rmse_g:.5f} | Test MAE={mae_g:.5f}")

    # Final preds
    model.eval()
    with torch.no_grad():
        Y_pred = model(torch.tensor(X_test_sc, dtype=torch.float32)).cpu().numpy()

    rmse_g, mae_g, rmse_by_m, mae_by_m = evaluate_metrics(Y_test, Y_pred)

    print("\n================ FINAL TEST METRICS ================")
    print(f"RMSE globale (flatten) : {rmse_g:.6f}")
    print(f"MAE  globale (flatten) : {mae_g:.6f}")

    print("\nRMSE par mois:")
    for m in range(1, 13):
        print(f"  mois {m:02d}: {rmse_by_m[m]:.6f}")

    print("\nMAE par mois:")
    for m in range(1, 13):
        print(f"  mois {m:02d}: {mae_by_m[m]:.6f}")

    return model, scaler, Y_pred


# =========================================================
# 6) PIPELINE COMPLET : AMÉNAGER LA BASE -> TRAIN -> EVAL
# =========================================================
# >>>>>> IMPORTANT <<<<<<


# 6.1) Définir "other" (variables avec lag1)
OTHER_COLS = THERMAL_COLS + HUMIDITY_COLS + PRECIP_COLS + EVAP_COLS + SNOW_COLS + FLOW_COLS

# 6.2) Créer les lags nécessaires
df_feat = df.copy()
df_feat = add_lags_panel(df_feat, SWI_UNIF_COLS, L_UNIF)
df_feat = add_lags_panel(df_feat, SWI_COLS,      L_SWI)
df_feat = add_lags_panel(df_feat, OTHER_COLS,    L_OTHER)

# 6.3) Construire la liste finale des features
X_cols = build_feature_columns(L_UNIF, L_SWI, OTHER_COLS, STATIC_COLS)
Y_cols = TARGET_COLS

# 6.4) Nettoyer uniquement sur X et Y (pas dropna global)
df_model = df_feat.dropna(subset=X_cols + Y_cols).reset_index(drop=True)

# 6.5) Split temporel
train_mask = df_model[TIME_COL] <= TRAIN_END
test_mask  = df_model[TIME_COL] >= TEST_START

X_all = df_model[X_cols].values.astype(np.float32)
Y_all = df_model[Y_cols].values.astype(np.float32)

X_train, Y_train = X_all[train_mask], Y_all[train_mask]
X_test,  Y_test  = X_all[test_mask],  Y_all[test_mask]

print("=================================================")
print("Data shapes")
print("X_train:", X_train.shape, "Y_train:", Y_train.shape)
print("X_test :", X_test.shape,  "Y_test :", Y_test.shape)
print("Nb features:", len(X_cols))
print("=================================================")

# 6.6) Entraîner + évaluer
model, scaler_X, Y_pred_test = train_nn_and_report(X_train, Y_train, X_test, Y_test)


Data shapes
X_train: (161916, 181) Y_train: (161916, 12)
X_test : (49460, 181) Y_test : (49460, 12)
Nb features: 181
Epoch 000 | Train MSE=0.01390 | Test RMSE=0.17460 | Test MAE=0.13201
Epoch 020 | Train MSE=0.00320 | Test RMSE=0.16909 | Test MAE=0.12616
Epoch 040 | Train MSE=0.00310 | Test RMSE=0.17126 | Test MAE=0.12761
Epoch 060 | Train MSE=0.00306 | Test RMSE=0.17084 | Test MAE=0.12709
Epoch 080 | Train MSE=0.00305 | Test RMSE=0.17483 | Test MAE=0.13065
Epoch 100 | Train MSE=0.00306 | Test RMSE=0.17149 | Test MAE=0.12811
Epoch 120 | Train MSE=0.00304 | Test RMSE=0.17249 | Test MAE=0.12957
Epoch 140 | Train MSE=0.00304 | Test RMSE=0.17039 | Test MAE=0.12694
Epoch 160 | Train MSE=0.00309 | Test RMSE=0.17603 | Test MAE=0.13021
Epoch 180 | Train MSE=0.00303 | Test RMSE=0.17140 | Test MAE=0.12786
Epoch 199 | Train MSE=0.00303 | Test RMSE=0.17153 | Test MAE=0.12741

================ FINAL TEST METRICS ================
RMSE globale (flatten) : 0.171532
MAE  globale (flatten) : 0.127409

R

In [5]:
# =========================================================
# AJOUT DES PRÉDICTIONS NN DANS LA BASE TEST COMPLÈTE
# =========================================================

# 1) Recréer la base test dans le même ordre que X_test / Y_pred_test
df_test = df_model.loc[test_mask].copy().reset_index(drop=True)

# 2) Noms des colonnes de prédiction (12 mois)
PRED_COLS = [f"swi_unif_pred_{str(m).zfill(2)}" for m in range(1, 13)]

# 3) Ajouter les prédictions
df_test[PRED_COLS] = Y_pred_test

# (optionnel) erreurs par mois
for m in range(1, 13):
    df_test[f"err_{m:02d}"] = (
        df_test[f"swi_unif_pred_{m:02d}"] -
        df_test[f"swi_unif_{m:02d}"]
    )

# Vérification rapide
print(df_test[PRED_COLS].head())


   swi_unif_pred_01  swi_unif_pred_02  swi_unif_pred_03  swi_unif_pred_04  \
0          0.954887          0.941240          0.908033          0.814562   
1          0.881813          0.925066          0.857099          0.742923   
2          0.961008          0.977374          0.899885          0.753785   
3          0.965714          0.943356          0.855914          0.743830   
4          0.952944          0.976522          0.863701          0.787387   

   swi_unif_pred_05  swi_unif_pred_06  swi_unif_pred_07  swi_unif_pred_08  \
0          0.650883          0.390610          0.163841          0.064742   
1          0.584564          0.311477          0.122346          0.042100   
2          0.713660          0.428264          0.253625          0.174550   
3          0.546767          0.288673          0.152427          0.089935   
4          0.731021          0.424015          0.187601          0.143358   

   swi_unif_pred_09  swi_unif_pred_10  swi_unif_pred_11  swi_unif_pred_12 

In [6]:
df_test.head()

,NUMERO,LAMBX_93,LAMBY_93,LAMBX_2,LAMBY_2,ANNEE,code_insee_commune,commune,code_insee_de_la_region,region,...,err_03,err_04,err_05,err_06,err_07,err_08,err_09,err_10,err_11,err_12
0,2,641374,7106309,588000,2673000,2020,59273,Gravelines,32,Hauts-de-France,...,-0.089967,0.110562,0.212883,0.159610,0.006841,-0.011258,-0.083043,-0.243193,-0.128475,-0.018287
1,2,641374,7106309,588000,2673000,2021,59273,Gravelines,32,Hauts-de-France,...,0.016099,0.054923,0.021564,-0.108523,-0.389654,-0.348900,-0.143110,-0.136188,-0.147091,-0.323303
2,2,641374,7106309,588000,2673000,2022,59273,Gravelines,32,Hauts-de-France,...,0.117885,0.019785,0.252660,0.192264,0.174625,0.185550,0.033145,-0.159283,-0.266363,-0.334924
3,2,641374,7106309,588000,2673000,2023,59273,Gravelines,32,Hauts-de-France,...,-0.057086,-0.191170,-0.216233,-0.105327,-0.046573,-0.108065,-0.041566,-0.024770,-0.529123,-0.409115
4,2,641374,7106309,588000,2673000,2024,59273,Gravelines,32,Hauts-de-France,...,-0.123299,-0.092613,-0.049979,-0.187985,-0.245399,-0.103642,-0.066050,-0.115388,0.090275,0.125864


In [7]:
# =========================================================
# SAUVEGARDE DE LA BASE df_test
# =========================================================

SAVE_PATH = "df_test_nn_predictions"

df_test.to_csv(f"{SAVE_PATH}.csv", index=False)
print("df_test sauvegardée avec succès.")


df_test sauvegardée avec succès.
